# 07 · Production Twin (Step 4)

The **Production twin** — the manufacturing network: what is planned, in progress, produced, and
where capacity/quality constraints emerge.

**Owns:** open production orders / WIP, planned vs actual, yield/scrap/rework, cost & schedule
variance, work-centre/bottleneck pressure.

**Inputs:** `gold.productionorders` (item) + `gold.productionordersheader` (header) + `gold.workcentres` (capacity + bottleneck flag) + `gold.routings` (operation standard values / work content).
**Outputs (monitoring):** `production_twin_current`, `production_twin_metrics_timeseries`.

Grain: material × plant. Per the data contract, production facts are filtered to header
`OrderCategory_AUTYP = '10'` and not-deleted. The routing join adds routed work-content minutes,
the primary work centre, and **work-backlog days** (open work content ÷ effective daily capacity)
as a capacity-pressure signal (`is_bottleneck` stays the master-data flag). _Setup-matrix
changeover sequencing not yet modelled (see readiness scan)._ Idempotent overwrite.

## Config

In [ ]:
dbutils.widgets.text("source_catalog", "mickmickstableawscoff_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold_opus5", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("window_days", "365", "Recent window for current-state rollups")

import uuid
from pyspark.sql import functions as F

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
WINDOW = int(dbutils.widgets.get("window_days"))
REFRESH_ID = str(uuid.uuid4())

PO   = f"{SRC}.productionorders"
POH  = f"{SRC}.productionordersheader"
WC   = f"{SRC}.workcentres"
ROUT = f"{SRC}.routings"
CUR_OUT = f"{FQ_MON}.production_twin_current"
TS_OUT  = f"{FQ_MON}.production_twin_metrics_timeseries"
print(f"Sources: {PO} + {POH} + {WC} + {ROUT}\nOutputs: {CUR_OUT} , {TS_OUT}\nWindow: {WINDOW}d")

## 1. Filtered production base (item + header + work centre)
Join items to headers, apply the `AUTYP='10'` / not-deleted contract, attach bottleneck flag.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW prod_base AS
SELECT po.Client_MANDT, po.ProductionOrderNumber_AUFNR, po.MaterialNumber_MATNR, po.Plant_WERKS,
       po.MaterialText_MAKTX, po.MaterialGroup_MATKL, po.BaseUoM_MEINS,
       po.PlannedQuantity_PSMNG, po.GoodsReceiptQuantity_WEMNG, po.OpenQuantity,
       po.ConfirmedScrapQuantity, po.ConfirmedReworkQuantity,
       po.YieldPercent, po.ScrapPercent, po.FirstPassYieldPercent,
       po.IsItemDelivered, po.ProductionDate, h.PrimaryWorkCentre_ARBID,
       h.CostVariancePercent, h.ScheduleAdherence, h.OnTimeStart,
       h.ActualLeadTimeHours, h.LeadTimeVarianceHours,
       wc.IsBottleneckFlag, wc.WorkCentreCode_ARBPL,
       rt.TotalStandardWorkContentMinutes, rt.WorkCentreCode_ARBPL AS routed_work_centre,
       rt.IsSchedulingRelevant, rwc.EffectiveDailyCapacityMinutes
FROM {PO} po
JOIN {POH} h
  ON po.Client_MANDT = h.Client_MANDT AND po.ProductionOrderNumber_AUFNR = h.ProductionOrderNumber_AUFNR
LEFT JOIN {WC} wc
  ON h.PrimaryWorkCentre_ARBID = wc.WorkCentreObjectID_OBJID AND po.Plant_WERKS = wc.Plant_WERKS
LEFT JOIN {ROUT} rt
  ON po.MaterialNumber_MATNR = rt.MaterialNumber_MATNR AND po.Plant_WERKS = rt.Plant_WERKS
LEFT JOIN {WC} rwc
  ON rt.WorkCentreObjectID_OBJID = rwc.WorkCentreObjectID_OBJID AND po.Plant_WERKS = rwc.Plant_WERKS
WHERE h.OrderCategory_AUTYP = '10' AND (h.DeletionFlag_LOEKZ IS NULL OR h.DeletionFlag_LOEKZ = '')
  AND po.MaterialNumber_MATNR IS NOT NULL
""")
print(f"prod_base rows: {spark.table('prod_base').count():,}")

## 2. Build `production_twin_current`
Recent-window manufacturing state per material × plant.

In [ ]:
current = spark.sql(f"""
SELECT *, (is_bottleneck_num > 0) AS is_bottleneck FROM (
  SELECT Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
    MAX(MaterialText_MAKTX) AS material_text, MAX(MaterialGroup_MATKL) AS material_group, MAX(BaseUoM_MEINS) AS uom,
    COUNT(DISTINCT ProductionOrderNumber_AUFNR) AS total_order_count,
    COUNT(DISTINCT CASE WHEN NOT IsItemDelivered THEN ProductionOrderNumber_AUFNR END) AS open_order_count,
    COALESCE(SUM(CASE WHEN NOT IsItemDelivered THEN OpenQuantity END), 0) AS open_production_qty,
    SUM(PlannedQuantity_PSMNG) AS planned_qty, SUM(GoodsReceiptQuantity_WEMNG) AS produced_qty,
    AVG(YieldPercent) AS yield_pct, AVG(ScrapPercent) AS scrap_pct, AVG(FirstPassYieldPercent) AS first_pass_yield_pct,
    SUM(ConfirmedScrapQuantity) AS scrap_qty, SUM(ConfirmedReworkQuantity) AS rework_qty,
    AVG(CostVariancePercent) AS avg_cost_variance_pct,
    AVG(CAST(ScheduleAdherence AS DOUBLE)) AS schedule_adherence_rate,
    AVG(CAST(OnTimeStart AS DOUBLE)) AS on_time_start_rate,
    AVG(ActualLeadTimeHours) AS avg_actual_lead_time_hours,
    AVG(LeadTimeVarianceHours) AS avg_lead_time_variance_hours,
    MAX(CAST(IsBottleneckFlag AS DOUBLE)) AS is_bottleneck_num,
    COALESCE(MAX(routed_work_centre), MAX(WorkCentreCode_ARBPL)) AS primary_work_centre,
    MAX(TotalStandardWorkContentMinutes) AS routed_work_minutes_per_unit,
    COALESCE(SUM(CASE WHEN NOT IsItemDelivered THEN OpenQuantity END), 0) * MAX(TotalStandardWorkContentMinutes) AS open_work_content_minutes,
    MAX(EffectiveDailyCapacityMinutes) AS work_centre_daily_capacity_min,
    CASE WHEN MAX(EffectiveDailyCapacityMinutes) > 0
         THEN (COALESCE(SUM(CASE WHEN NOT IsItemDelivered THEN OpenQuantity END), 0) * MAX(TotalStandardWorkContentMinutes)) / MAX(EffectiveDailyCapacityMinutes) END AS work_backlog_days,
    CAST((SELECT MAX(ProductionDate) FROM prod_base) AS DATE) AS as_of_date
  FROM prod_base
  WHERE ProductionDate > date_sub((SELECT MAX(ProductionDate) FROM prod_base), {WINDOW})
  GROUP BY 1,2,3
)
""")
current = (current.drop("is_bottleneck_num")
    .withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(current.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CUR_OUT))
spark.sql(f"COMMENT ON TABLE {CUR_OUT} IS 'Production twin current state per material x plant: WIP/open orders, planned vs produced, yield/scrap/rework, cost & schedule variance, bottleneck pressure.'")
print(f"Wrote {current.count():,} material-plant rows to {CUR_OUT}")

## 3. Build `production_twin_metrics_timeseries`
Manufacturing output & quality over time (weekly by production date).

In [ ]:
timeseries = spark.sql(f"""
SELECT Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
       date_trunc('week', ProductionDate) AS period_start,
  COUNT(DISTINCT ProductionOrderNumber_AUFNR) AS order_count,
  SUM(PlannedQuantity_PSMNG) AS planned_qty, SUM(GoodsReceiptQuantity_WEMNG) AS produced_qty,
  AVG(YieldPercent) AS yield_pct, AVG(ScrapPercent) AS scrap_pct,
  SUM(ConfirmedScrapQuantity) AS scrap_qty, SUM(ConfirmedReworkQuantity) AS rework_qty
FROM prod_base WHERE ProductionDate IS NOT NULL
GROUP BY 1,2,3,4
""")
timeseries = (timeseries.withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(timeseries.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TS_OUT))
spark.sql(f"COMMENT ON TABLE {TS_OUT} IS 'Production twin metric time-series per material x plant x week: planned/produced qty, yield/scrap, scrap/rework qty, order count.'")
print(f"Wrote {timeseries.count():,} rows to {TS_OUT}")

## Column descriptions
Apply human-readable descriptions to every output column (persisted on rebuild).

In [ ]:
# Column descriptions — re-applied after the overwrite writes so they persist in the
# catalog on every rebuild. EXTEND for your business.
COLUMN_COMMENTS = {
    CUR_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "material_text": "Material description",
        "material_group": "Material group / category",
        "uom": "Base unit of measure",
        "total_order_count": "Distinct production orders in the window",
        "open_order_count": "Distinct production orders not yet delivered",
        "open_production_qty": "Open (not yet produced) quantity on not-yet-delivered order items (residuals on closed orders excluded)",
        "planned_qty": "Total planned production quantity",
        "produced_qty": "Total produced (goods-receipt) quantity",
        "yield_pct": "Average yield percent",
        "scrap_pct": "Average scrap percent",
        "first_pass_yield_pct": "Average first-pass yield percent",
        "scrap_qty": "Total confirmed scrap quantity",
        "rework_qty": "Total confirmed rework quantity",
        "avg_cost_variance_pct": "Average cost variance percent (actual vs planned)",
        "schedule_adherence_rate": "Share of orders meeting schedule",
        "on_time_start_rate": "Share of orders starting on time",
        "avg_actual_lead_time_hours": "Average actual production lead time in hours",
        "avg_lead_time_variance_hours": "Average lead-time variance vs plan in hours",
        "primary_work_centre": "Primary work centre from the routing (falls back to the order header)",
        "routed_work_minutes_per_unit": "Standard work-content minutes per unit from the routing",
        "open_work_content_minutes": "Open quantity multiplied by routed minutes per unit (work backlog)",
        "work_centre_daily_capacity_min": "Effective daily capacity of the primary work centre in minutes",
        "work_backlog_days": "Open work-content minutes divided by daily capacity = days to clear open work at full capacity (informational capacity-pressure signal)",
        "as_of_date": "As-of date of this twin snapshot",
        "is_bottleneck": "True when the primary work centre is flagged as a bottleneck in master data",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
    TS_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "period_start": "Start of the week bucket (production date)",
        "order_count": "Distinct production orders in the week",
        "planned_qty": "Planned quantity in the week",
        "produced_qty": "Produced quantity in the week",
        "yield_pct": "Average yield percent in the week",
        "scrap_pct": "Average scrap percent in the week",
        "scrap_qty": "Confirmed scrap quantity in the week",
        "rework_qty": "Confirmed rework quantity in the week",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
}
for _tbl, _cols in COLUMN_COMMENTS.items():
    for _c, _desc in _cols.items():
        spark.sql(f"ALTER TABLE {_tbl} ALTER COLUMN {_c} COMMENT '{_desc}'")
print(f"Applied column descriptions to {CUR_OUT} and {TS_OUT}")

## 4. Verify

In [ ]:
print("=== production_twin_current (sample) ===")
display(spark.sql(f"""
SELECT material, plant, material_group, open_order_count, open_production_qty,
       ROUND(yield_pct,1) yield_pct, ROUND(scrap_pct,1) scrap_pct, ROUND(avg_cost_variance_pct,1) cost_var,
       ROUND(schedule_adherence_rate,2) sched_adh, primary_work_centre,
       ROUND(routed_work_minutes_per_unit,1) routed_min, ROUND(work_backlog_days,1) backlog_days, is_bottleneck
FROM {CUR_OUT} ORDER BY open_production_qty DESC NULLS LAST LIMIT 25
"""))
display(spark.sql(f"SELECT COUNT(*) rows, SUM(CAST(is_bottleneck AS INT)) bottleneck_items, ROUND(AVG(yield_pct),1) avg_yield, ROUND(AVG(scrap_pct),1) avg_scrap FROM {CUR_OUT}"))
display(spark.sql(f"SELECT COUNT(*) rows, COUNT(DISTINCT material||plant) series, MIN(period_start) first_p, MAX(period_start) last_p FROM {TS_OUT}"))

## Next
Production twin live. Next: **Distribution** (`08_distribution_twin`) — outbound fulfillment,
OTIF, fill rate, delivery delay, fulfillment risk.